# **Calsificador de Reseñas**

Este modelo tiene como propósito generar un clasificador de reseñas, principalmente de productos de Amazon. De esta forma al colocar una reseña este modelo clasificara si es:
  1. Positiva
  2. Negativa

Se decidio solamente tomar en cuenta estos 2 elementos porque de esta forma ayudara al modelo a ser más preciso, pues si consideramos las 5 entrellas que podrias tener el producto, afectaria los pronosticos al estas desvalanceado y la matorias de las veces asignaira o 5 o 1 estrella.

El dataset se tomo de kaggle, siendo el siguiente:

- [Dataset Reseñas de Amazon](https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews)

Para generar un modelo, primero se hizo el siguiente ETL del modelo:


# **1. Cargar el Dataset**

In [2]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Se sube el archivo csv en un dataframe de pandas, para poder realizar el tratamietno de datos especifico para que funcione correctamente.

In [4]:
import pandas as pd

file_path = '/content/drive/MyDrive/Desarrollo de aplicaciones avanzadas de ciencias computacionales/Embeddings/Data/Reviews.csv'

try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file was not found at {file_path}")
except Exception as e:
    print(f"An error occurred: {e}")

Dataset loaded successfully!


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


Se revisa el tamaño de cada estrella de la sereña para poder verificar que tan balanceado esta.

In [5]:
df['Score'].value_counts()

,count
Score,
5,363122
4,80655
1,52268
3,42640
2,29769


Como podemos observar el dataset en su mayoria tiene reseñas de 5 estrellas o más por lo que sin duda va afectar al rendimiento del modelo, por eso se decidio colocar una etiqueta nueva donde solamente diga si es resela negativa o positiva.

In [6]:
def new_label(score):
    if (score>=4):
        return 1
    elif (score<=3):
        return 0
    else:
        return None

df['label'] = df['Score'].apply(lambda x: new_label(x))
df['label'].unique()

df['label'].value_counts()

,count
label,
1,443777
0,124677


Se elimina las columnas que estén vacias en "label".

In [7]:
df.dropna(subset='label', inplace= True)

Se quitan los caracteres alfa numéricos y se pasa todo a minusculas para no clasificar palabras iguales o que no tengan significado, para ello se crea una nueva columna con los nuevos cambios para comparacion.

In [15]:
import re

df['Clean Text'] = df['Text'].str.lower()
df['Clean Text'] = df['Clean Text'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x))

Ahora por medio de una lista de stopwords obtenida de github en fromato txt, siento la siguiente: [stopwords-en.txt](https://github.com/stopwords-iso/stopwords-en/blob/master/stopwords-en.txt).

Se cargo y se colocó en una lista para de esta forma se revise en cada oración si hay un stopword y se elimina, para tener de esta forma oraciones con palabras significativas

In [16]:
file_stopwords = '/content/drive/MyDrive/Desarrollo de aplicaciones avanzadas de ciencias computacionales/Embeddings/Stopwords/stopwords-en.txt'

with open(file_stopwords, "r", encoding="utf-8") as f:
    stopwords = set(line.strip() for line in f if line.strip())

df['Clean Text'] = df['Clean Text'].apply(
    lambda text: ' '.join(word for word in text.split() if word.lower() not in stopwords)
)

Se eliminan las filas que no tiene nada en la columna 'Clean Text', pues significa que no tenia unformación relevante.

In [18]:
df.dropna(subset='Clean Text', inplace= True)

df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,label,Clean Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...,1,bought vitality canned dog food products quali...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,0,product arrived labeled jumbo salted peanutsth...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...,1,confection centuries light pillowy citrus gela...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...,0,secret ingredient robitussin addition root bee...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...,1,taffy price wide assortment yummy taffy delive...
...,...,...,...,...,...,...,...,...,...,...,...,...
568449,568450,B001EO7N10,A28KG5XORO54AY,Lettie D. Carter,0,0,5,1299628800,Will not do without,Great for sesame chicken..this is a good if no...,1,sesame chickenthis resturants eaten atmy husba...
568450,568451,B003S1WTCU,A3I8AFVPEE8KI5,R. Sawyer,0,0,2,1331251200,disappointed,I'm disappointed with the flavor. The chocolat...,0,disappointed flavor chocolate notes weak milk ...
568451,568452,B004I613EE,A121AA1GQV751Z,"pksd ""pk_007""",2,2,5,1329782400,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o...",1,stars training session train dog ceaser dog tr...
568452,568453,B004I613EE,A3IBEVCTXKNOH,"Kathy A. Welch ""katwel""",1,1,5,1331596800,Favorite Training and reward treat,These are the BEST treats for training and rew...,1,treats training rewarding dog grooming calorie...


# **2. Transformación de datos**

Primero revisamos el desvalanceo de las muestras

In [14]:
counts = df["label"].value_counts().sort_values(ascending=False)
perc   = (counts / counts.sum() * 100).round(2)

balance_tbl = pd.DataFrame({"count": counts, "percent": perc})
display(balance_tbl)
print("Total muestras:", counts.sum())

,count,percent
label,,
1,443777,78.07
0,124677,21.93


Total muestras: 568454


Como se puede ver esta bastante desvalanceado entronses en problable que aprenda mas 1 que 0, sin emabrgo vamos a evaluar el modelo y si tiene mérticas bajas se tratara de igualar con otro dataset o eliminado filas

# **3. Separación del DataFrame**

Se separa el dataframe en 'X' y 'y' para su clasificación quedando de esta forma:
- X = texto_limpio (string)
- y = etiqueta (int)

Una vez limpio se separa por medio de la libreria de sklearn llamada  train_test_split, con esto se crea el validation, test y train quedando de esta froma la distribución:
- train 80%
- test 10%
- validation 10%

Con esto nos aseguramos que tenga una distribución uniforme todos los sets para un entrenamiento y pruebas correctas.

In [24]:
from sklearn.model_selection import train_test_split

X = df['Clean Text']
y = df['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

In [25]:
train_df = pd.DataFrame({'Clean Text': X_train, 'label': y_train})
muestra = train_df.sample(10, random_state=42)

for _, row in muestra.iterrows():
    print("Question:", row["Clean Text"])
    print("Label:", row["label"])

Question: cheezit mozzarella cheese snack totally yummy entire family loved fans regular cheezits flavor mild tasty cheezit mozzarella cheese flavor flavor original cheese family eat
Label: 1
Question: coffee reorder start day coffee
Label: 1
Question: quality chews dog loves teeth breath clean fresh highly recommend sale
Label: 1
Question: almonds totally addictive deeeeliciousbr deal costs stores
Label: 1
Question: gained control rights iconic bridge manhattan boroughs invest portion unique property respond nespresso capsules
Label: 0
Question: starting wine knowledge happy yeast produced wait age bit nice esters wine
Label: 1
Question: nice smooth flavor adding heavy wipping cream condensed milk tea drinking experience nicer
Label: 1
Question: love strawberry honey bunches oats cereal eater love discovered peach love hey love pecansbr nope cinnamon flavor cinnamon people flavor cereal
Label: 0
Question: tested bottles essentia swanson tape testednot close advertised disappointing
La

# **4. Vectorización del texto**

Ahora se hace es una vectorización del DataFrame, Para eso se uso la libreria TfidfVectorizer el cual usará el método estadístico TF-IDF, siendo muy util para evitar palabras que pueden llegar a ser muy repetitivas o que no sean tan usadas pues de esta forma se puede evitar el uso de stopwords que se puedieran pasar o palabras que sean muy raras y aaprezcan en menos de 2 oraciones pues puede ser que esten mal escritas asi que para no entrar en conflicto se decide elmininar

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

VOCAB_SIZE = 50000
MAX_FEATURES = VOCAB_SIZE

tfidf = TfidfVectorizer(
    max_features=MAX_FEATURES,
    max_df=0.95,
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

# **5. Guardar DF y vocabularios**

Ahora con lo obtenido se guardan los datos, se usa matrices sparse para guardarlos pues de esta forma se guardan al usar tfidf.

In [28]:
import os
import scipy.sparse as sp

base_dir = "/content/drive/MyDrive/Desarrollo de aplicaciones avanzadas de ciencias computacionales/Embeddings/Data"
os.makedirs(base_dir, exist_ok=True)

train_dir = os.path.join(base_dir, "train_dataset")
val_dir   = os.path.join(base_dir, "val_dataset")
test_dir  = os.path.join(base_dir, "test_dataset")

sp.save_npz(os.path.join(base_dir, "X_train_tfidf.npz"), X_train_tfidf)
sp.save_npz(os.path.join(base_dir, "X_val_tfidf.npz"), X_val_tfidf)
sp.save_npz(os.path.join(base_dir, "X_test_tfidf.npz"), X_test_tfidf)

print(f" Datasets guardados correctamente en:\n {base_dir}")

 Datasets guardados correctamente en:
 /content/drive/MyDrive/Desarrollo de aplicaciones avanzadas de ciencias computacionales/Embeddings/Data


Tambien se guardan el vocabulario en un formato JSON para su futuro uso y toke nizar futuramente las palabras para testeos personalizados

In [31]:
import json

with open(os.path.join(base_dir, "vocab.json"), "w") as f:
    json.dump({k: int(v) for k, v in tfidf.vocabulary_.items()}, f)